In [4]:
import pandas as pd 
from sqlalchemy import create_engine

In [5]:
DB_USER = 'postgres'
DB_PASSWORD = 'Dikshu1998'
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'olist_db'

In [7]:
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [8]:
# 2. Map of Table Names to File Names
dataset_files = {
    'olist_customers': 'olist_customers_dataset.csv',
    'olist_geolocation': 'olist_geolocation_dataset.csv',
    'olist_orders': 'olist_orders_dataset.csv',
    'olist_order_items': 'olist_order_items_dataset.csv',
    'olist_order_payments': 'olist_order_payments_dataset.csv',
    'olist_order_reviews': 'olist_order_reviews_dataset.csv',
    'olist_products': 'olist_products_dataset.csv',
    'olist_sellers': 'olist_sellers_dataset.csv',
    'product_category_name_translation': 'product_category_name_translation.csv'
}

In [9]:
# Date columns in olist_orders to parse as timestamps
orders_date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

In [10]:
# Date columns in olist_order_reviews to parse as timestamps
reviews_date_cols = [
    'review_creation_date',
    'review_answer_timestamp'
]

In [12]:
# 3. Ingestion Loop
print("Starting full data ingestion into PostgreSQL...\n")


for table_name, file_path in dataset_files.items():
    print(f"Reading {file_path}...")
    df = pd.read_csv(file_path)
    
    # Handle timestamp conversions
    if table_name == 'olist_orders':
        for col in orders_date_cols:
            df[col] = pd.to_datetime(df[col])
            
    elif table_name == 'olist_order_reviews':
        for col in reviews_date_cols:
            df[col] = pd.to_datetime(df[col])
            
    elif table_name == 'olist_order_items':
        df['shipping_limit_date'] = pd.to_datetime(df['shipping_limit_date'])
    # Write to PostgreSQL
    print(f"Loading into PostgreSQL table '{table_name}'...")
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f"Loaded {len(df):,} rows into '{table_name}'.\n")

print("All 9 datasets have been successfully ingested into PostgreSQL!")

Starting full data ingestion into PostgreSQL...

Reading olist_customers_dataset.csv...
Loading into PostgreSQL table 'olist_customers'...
Loaded 99,441 rows into 'olist_customers'.

Reading olist_geolocation_dataset.csv...
Loading into PostgreSQL table 'olist_geolocation'...
Loaded 1,000,163 rows into 'olist_geolocation'.

Reading olist_orders_dataset.csv...
Loading into PostgreSQL table 'olist_orders'...
Loaded 99,441 rows into 'olist_orders'.

Reading olist_order_items_dataset.csv...
Loading into PostgreSQL table 'olist_order_items'...
Loaded 112,650 rows into 'olist_order_items'.

Reading olist_order_payments_dataset.csv...
Loading into PostgreSQL table 'olist_order_payments'...
Loaded 103,886 rows into 'olist_order_payments'.

Reading olist_order_reviews_dataset.csv...
Loading into PostgreSQL table 'olist_order_reviews'...
Loaded 99,224 rows into 'olist_order_reviews'.

Reading olist_products_dataset.csv...
Loading into PostgreSQL table 'olist_products'...
Loaded 32,951 rows into 